In [11]:
import os, re, numpy as np
import mne
import gc    
from datetime import timedelta
import pandas as pd

import matplotlib.pyplot as plt
import pywt
from scipy.signal import hilbert
import networkx as nx
from pathlib import Path
from scipy.stats import entropy
from scipy.signal import butter, filtfilt
from mne.filter import filter_data, notch_filter

In [12]:
# ==========================================================================
# 0. SAMPLING FREQUENCY AND WAVEBANDS CONSTANTS
# ==========================================================================
FS = 256

BANDS = {'delta':[0.5, 4.0],
        'theta': [4.0, 8.0],
        'alpha':[8.0, 13.0],
        'beta': [13.0, 30.0],
        'gamma': [30.0, 100]}

In [13]:
# ==========================================================================
# 1. CONNECTIVITY ESTIMATION FUNCTIONS 
# ==========================================================================
# wPLI
'''
def get_matrix(analytic_signal):
    n_ch = analytic_signal.shape[0]
    phase = np.angle(analytic_signal)  # shape (n_ch, n_times)
    wpli_matrix = np.zeros((n_ch, n_ch))
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            delta_phi = phase[i] - phase[j]
            sin_dphi = np.sin(delta_phi)

            wpli = np.abs(np.mean(sin_dphi)) / (np.mean(np.abs(sin_dphi)) + 1e-10)
            wpli_matrix[i, j] = wpli_matrix[j, i] = wpli

    return wpli_matrix'''

#PLI
def get_matrix(analytic_signal):
    n_ch = analytic_signal.shape[0]
    
    phase = np.angle(analytic_signal)

    pli_matrix = np.zeros((n_ch, n_ch))
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            phase_diff = phase[i] - phase[j]
            pli = np.abs(np.mean(np.sign(np.sin(phase_diff))))
            pli_matrix[i, j] = pli_matrix[j, i] = pli
    return pli_matrix

'# wPLI\ndef get_matrix(analytic_signal):\n    n_ch = analytic_signal.shape[0]\n    phase = np.angle(analytic_signal)  # shape (n_ch, n_times)\n    wpli_matrix = np.zeros((n_ch, n_ch))\n    for i in range(n_ch):\n        for j in range(i + 1, n_ch):\n            delta_phi = phase[i] - phase[j]\n            sin_dphi = np.sin(delta_phi)\n\n            wpli = np.abs(np.mean(sin_dphi)) / (np.mean(np.abs(sin_dphi)) + 1e-10)\n            wpli_matrix[i, j] = wpli_matrix[j, i] = wpli\n\n    return wpli_matrix'

In [ ]:
mode = 'PLI'

input_path = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-segment4h-testing'
output_base = f'/kaggle/working/CHB-MIT_connectivity/{mode}_matrix'

for chb in sorted(os.listdir(input_path)):
    chb_path = os.path.join(input_path, chb, 'testing')
    if not os.path.isdir(chb_path) or chb == f'{mode}_matrix':
        continue
    print(f'Processing {chb}')
    seizures = os.listdir(chb_path)
    for sz in sorted(seizures):
        sz_path = os.path.join(chb_path, sz)
        if not os.path.isdir(sz_path):
            continue
        for batch in sorted(os.listdir(sz_path)):
            if batch.endswith(".npy"):
                full_path = os.path.join(sz_path, batch)
                epochs = np.load(full_path)  # shape (n_epochs, n_channels, n_samples)
                batch_idx = batch.split('.')[0]
                epoch_output = os.path.join(output_base, chb, sz, batch_idx)
                os.makedirs(epoch_output, exist_ok=True)
                for band, (lowcut, highcut) in BANDS.items():
                    epoch_list = []
                    for epoch_idx, epoch in enumerate(epochs):
                        epoch = filter_data(epoch, sfreq=FS, l_freq=lowcut, h_freq=highcut, method='fir', fir_design='firwin', verbose=False)
                        analytic_epoch = hilbert(epoch, axis=-1)
                        
                        epoch_matrix = get_matrix(analytic_epoch)
                        epoch_list.append(epoch_matrix)
                    
                    epoch_array = np.array(epoch_list)
                    
                    save_file_path = os.path.join(epoch_output, f"{band}.npy")
                    np.save(save_file_path, epoch_array)
                    print(f'  [OK] {chb} | Seizure {sz}| Batch {batch_idx} | {band}.npy -> {epoch_array.shape}')
                    del epoch_list
                    del epoch_array
                    gc.collect()

                del epochs
                gc.collect()

Processing chb13
  [OK] chb13 | Seizure 1| Batch 0 | delta.npy -> (1800, 18, 18)
  [OK] chb13 | Seizure 1| Batch 0 | theta.npy -> (1800, 18, 18)
  [OK] chb13 | Seizure 1| Batch 0 | alpha.npy -> (1800, 18, 18)
  [OK] chb13 | Seizure 1| Batch 0 | beta.npy -> (1800, 18, 18)


In [ ]:
mode = 'wPLI'

input_path = '/kaggle/input/datasets/ngoclannguyen00/chb-mit-segment4h-all'
output_base = f'/kaggle/working/CHB-MIT_connectivity/{mode}_matrix'

for chb in sorted(os.listdir(input_path)):
    chb_path = os.path.join(input_path, chb, 'Training')
    if not os.path.isdir(chb_path) or chb == f'{mode}_matrix':
        continue
    print(f'Processing {chb}')
    labels = os.listdir(chb_path)
    for label in sorted(labels):
        label_path = os.path.join(chb_path, label)
        if not os.path.isdir(label_path):
            continue
        interval_name = label
        for file in sorted(os.listdir(label_path)):
            if file.endswith(".npy"):
                full_path = os.path.join(label_path, file)
                epochs = np.load(full_path)  # shape (n_epochs, n_channels, n_samples)
                file_name = file.split('.')[0]
                for band, (lowcut, highcut) in BANDS.items():
                    epoch_output = os.path.join(output_base, chb, interval_name, file_name)
                    os.makedirs(epoch_output, exist_ok=True)

                    epoch_list = []
                    for epoch_idx, epoch in enumerate(epochs):
                        epoch = filter_data(epoch, sfreq=FS, l_freq=lowcut, h_freq=highcut, method='fir', fir_design='firwin', verbose=False)
                        analytic_epoch = hilbert(epoch, axis=-1)
                        
                        epoch_matrix = get_matrix(analytic_epoch)
                        epoch_list.append(epoch_matrix)
                    
                    epoch_array = np.array(epoch_list)
                    
                    save_file_path = os.path.join(epoch_output, f"{band}.npy")
                    np.save(save_file_path, epoch_array)
                    if interval_name == ' preictal':
                        print(f'  [OK] {chb} | {interval_name}| seizure {file_name} | {band}.npy -> {epoch_array.shape}')
                    else:
                        print(f'  [OK] {chb} | {interval_name}| Batch {file_name} | {band}.npy -> {epoch_array.shape}')
                    del epoch_list
                    del epoch_array
                    gc.collect()

                del epochs
                gc.collect()
